In [ ]:
# Phase 1 REAL submission: loads the trained fold-0 checkpoint (efficientnet_b0,
# single sagittal fluid-sensitive series, 16 slices, lexical-label baseline,
# OOF macro AUC 0.7985 across all 5 folds) and runs it over the real hidden
# test set. Internet is OFF (competition rule for the actual submission --
# unlike the training notebook, which had internet on for ImageNet weights).
import glob, os, shutil, sys, time

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
ckpt_candidates = glob.glob('/kaggle/input/**/knee_phase1_fold0.pt', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
CKPT_PATH = ckpt_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import knee.dataset as knee_dataset
import knee.model as knee_model
import knee.infer as knee_infer
print('knee package imported successfully from', PKG)
print('checkpoint found at', CKPT_PATH)

In [ ]:
import pandas as pd
import torch

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
print(f'{len(test_df)} test studies')

device = 'cpu'
if torch.cuda.is_available():
    try:
        _probe = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as e:
        print(f'GPU present but incompatible with this PyTorch build: {e}')
print('device:', device)

model = knee_model.KneeModel(backbone_name='efficientnet_b0', num_labels=len(knee_infer.LABEL_COLUMNS), pretrained=False)
# weights_only=True: this checkpoint is our own (torch.save(model.state_dict(), ...)
# from the training notebook, tensors only), but torch.load defaults to
# weights_only=False which unpickles arbitrary objects -- no reason to allow
# that for a plain state_dict, so pin the safer loader explicitly.
state_dict = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print('trained checkpoint loaded')

# The model's 12 output logits come from one shared Linear layer. Only the
# 4 labels with lexical-rule coverage during training (ACL, Medial Meniscus,
# Effusion, Baker's) ever contributed to the loss -- the other 8 rows of that
# layer's weight matrix never received a gradient and are still at random
# initialization. Clamp those to the 0.5 benchmark floor rather than emit an
# untrained, meaningless prediction -- a constant beats an actively-wrong one.
SUPPORTED_LABELS = {'ACL', 'Medial Meniscus', 'Effusion', "Baker's"}
UNSUPPORTED_IDX = [i for i, l in enumerate(knee_infer.LABEL_COLUMNS) if l not in SUPPORTED_LABELS]
print('unsupported (clamped to 0.5) label indices:', UNSUPPORTED_IDX)

In [ ]:
import numpy as np

dataset = knee_dataset.KneeStudyDataset(
    study_uids=test_df['StudyInstanceUID'].tolist(),
    dcm_root=f'{COMP_DIR}/test_series',
    series_df=test_series_df,
    n_slices=16,
    size=224,
    max_series=1,
)

per_study_seconds = []

def predict_one(study_uid):
    idx = dataset.study_uids.index(study_uid)
    t0 = time.time()
    image, _, _ = dataset[idx]
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        probs = torch.sigmoid(logits)[0].cpu().numpy()
    probs[UNSUPPORTED_IDX] = 0.5
    per_study_seconds.append(time.time() - t0)
    return probs

wall_start = time.time()
submission = knee_infer.build_submission(test_df['StudyInstanceUID'].tolist(), predict_one)
wall_total = time.time() - wall_start

print(f'total wall time for {len(test_df)} studies: {wall_total:.2f}s ({wall_total/60:.1f} min)')
if per_study_seconds:
    print(f'mean per-study time: {np.mean(per_study_seconds):.3f}s')
print(f'studies that hit the 0.5 fallback (decode failure): {len(test_df) - len(per_study_seconds)}')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission)

# sanity checks before this becomes a real competition submission
assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert submission[knee_infer.LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[knee_infer.LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'all predictions must be in [0, 1]'
print('submission.csv passed shape/range checks')